## INM705 Deep Learning for Image Analysis
#### Object detection with YOLO
#### Author: Bo Fu, Yehoshua Perez Condori

In [ ]:
from modules.Dataset import get_dataloaders
from modules.Train import train
from modules.Evaluation import evaluate
from modules.Models.YOLOv1 import YOLOv1
from config import CKPT_DIR, DEVICE
import torch

In [ ]:
# YOLO parameters
S = 7
B = 2
C = 20

# training parameters
BATCH_SIZE   = 16 
EPOCHS       = 10 #60
LR           = 1e-3
WEIGHT_DECAY = 1e-4

# loss weights
LAMBDA_BOX   = 5.0
LAMBDA_NOOBJ = 0.5

# inference thresholds
CONF_THRESH    = 0.30
NMS_IOU_THRESH = 0.45

# experiment name
RUN_NAME = "baseline_10ep"

In [ ]:
# create model
model = YOLOv1(S=S, B=B, C=C).to(DEVICE)

# multi-GPU support
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = torch.nn.DataParallel(model)

# unwrap DataParallel
raw_model = model.module if isinstance(model, torch.nn.DataParallel) else model


In [ ]:
# create all data loaders
train_loader, val_loader, test_loader = get_dataloaders(BATCH_SIZE, S, B, C)

In [ ]:
# training
raw_model = train(
    model=model,
    raw_model=raw_model,
    train_loader=train_loader,
    val_loader=val_loader,
    S=S, B=B, C=C,
    BATCH_SIZE=BATCH_SIZE,
    EPOCHS=EPOCHS,
    LR=LR,
    WEIGHT_DECAY=WEIGHT_DECAY,
    LAMBDA_BOX=LAMBDA_BOX,
    LAMBDA_NOOBJ=LAMBDA_NOOBJ,
    RUN_NAME=RUN_NAME
)

# save trained weights
torch.save(raw_model.state_dict(), f"{CKPT_DIR}/{RUN_NAME}.pth")
print(f"trained weights saved: {RUN_NAME}.pth")

In [ ]:
# evaluation

# load trained weights before evaluation
raw_model.load_state_dict(torch.load(f"{CKPT_DIR}/{RUN_NAME}.pth", map_location=DEVICE))

results = evaluate(
    model=model,
    test_loader=test_loader,
    S=S, B=B, C=C,
    conf_thresh=CONF_THRESH,
    iou_thresh=NMS_IOU_THRESH
)